# Market data: curves and `MarketContext`

**Prerequisites:** complete notebooks **01** and **02** in this curriculum (finstack foundations: environment, core types, and dates/money basics). This notebook assumes you can import `finstack` and work with `datetime.date` and `Currency`.

## Concepts

Pricing and risk need a **consistent snapshot of the market** on a valuation date. In finstack, scalar term structures—discount factors, forward rates, credit hazards—are represented as **curves** keyed by time (year fractions from a **base date**).

The **`MarketContext`** container is the single place you **register** those curves (and FX) so downstream pricers can **`get_discount`**, **`get_forward`**, **`get_hazard`**, and read **`mc.fx`** without threading dozens of objects through every call.

- **DiscountCurve** — risk-free (or funding) discount factors \(DF(t)\); related **zero** rates via \(DF(t) = e^{-z(t)\,t}\).
- **ForwardCurve** — simple forward rates for a fixed **reset tenor** (e.g. 3M SOFR).
- **HazardCurve** — credit: piecewise hazard rates → survival probabilities for default timing.
- **FxMatrix** — stored FX quotes; **`FxConversionPolicy`** selects how a rate is applied for a given valuation date.

Below we walk through each API briefly, then assemble everything for a **USD corporate bond** style setup (OIS discount, SOFR forward, issuer hazard, cross-currency FX).

### DiscountCurve

Build from an identifier, **base date**, list of **(time in years, discount factor)** knots, and a **day-count** string. Use **`df(t)`** and **`zero(t)`** on the same time grid the curve uses internally (year fractions).

In [ ]:
from datetime import date

from finstack.core.market_data import DiscountCurve

curve = DiscountCurve(
    "USD-OIS",
    date(2024, 1, 1),
    [(0.0, 1.0), (0.5, 0.975), (1.0, 0.95), (2.0, 0.90), (5.0, 0.75), (10.0, 0.50)],
    day_count="act_365f",
)
print("id:", curve.id)
print("base_date:", curve.base_date)
print("df(1.0):", curve.df(1.0))
print("zero(1.0):", curve.zero(1.0))

### ForwardCurve

Forward curves are defined by **reset tenor in years** (e.g. `0.25` for three months), **knots** `(t, forward_rate)`, **`base_date`**, and **day_count**.

In [ ]:
from datetime import date

from finstack.core.market_data import ForwardCurve

fwd = ForwardCurve(
    "USD-SOFR",
    0.25,
    [(0.0, 0.04), (1.0, 0.045), (2.0, 0.048), (5.0, 0.05)],
    base_date=date(2024, 1, 1),
    day_count="act_360",
)
print("id:", fwd.id)
print("rate(1.0):", fwd.rate(1.0))
print("rate(3.0) interpolated:", fwd.rate(3.0))

### HazardCurve (credit)

If your build exposes **`HazardCurve`**, you can model issuer default intensity over time. Knots are **(time, hazard rate)**; optional **recovery** and **day_count** match the Rust builder. If the symbol is missing in your environment, the cell reports that and continues.

In [ ]:
from datetime import date

try:
    from finstack.core.market_data import HazardCurve

    hz = HazardCurve(
        "ACME-HZD",
        date(2024, 1, 1),
        [(1.0, 0.01), (5.0, 0.03), (10.0, 0.05)],
    )
    print("HazardCurve id:", hz.id)
    print("survival(5.0):", hz.survival(5.0))
    print("hazard_rate(5.0):", hz.hazard_rate(5.0))
except ImportError as e:
    print("HazardCurve not available:", e)

### FxMatrix

Store quotes with **`set_quote(base, quote, rate)`** meaning **1 base = rate quote**. Retrieve with **`rate(base, quote, as_of_date, policy)`**; the result exposes **`.rate`**. Cross pairs are **not** automatically triangulated through USD in current parity tests—you need a **direct** quote for pairs like GBP/EUR if you want a lookup to succeed.

In [ ]:
from datetime import date

from finstack.core.currency import Currency
from finstack.core.market_data import FxConversionPolicy, FxMatrix

fx = FxMatrix()
fx.set_quote(Currency("EUR"), Currency("USD"), 1.10)
fx.set_quote(Currency("GBP"), Currency("USD"), 1.27)
fx.set_quote(Currency("JPY"), Currency("USD"), 0.0067)

as_of = date(2024, 1, 1)
eur_usd = fx.rate(
    Currency("EUR"), Currency("USD"), as_of, FxConversionPolicy.CASHFLOW_DATE
)
print("EUR/USD:", eur_usd.rate)
jpy_usd = fx.rate(
    Currency("JPY"), Currency("USD"), as_of, FxConversionPolicy.CASHFLOW_DATE
)
print("JPY/USD:", jpy_usd.rate)

try:
    cross = fx.rate(
        Currency("GBP"), Currency("EUR"), as_of, FxConversionPolicy.CASHFLOW_DATE
    )
    print("GBP/EUR (unexpected without direct quote):", cross.rate)
except Exception as e:
    print("GBP/EUR without direct quote:", type(e).__name__, "—", e)

fx.set_quote(Currency("GBP"), Currency("EUR"), 1.27 / 1.10)
gbp_eur = fx.rate(
    Currency("GBP"), Currency("EUR"), as_of, FxConversionPolicy.CASHFLOW_DATE
)
print("GBP/EUR after direct quote:", gbp_eur.rate)

### MarketContext

**`insert(curve)`** registers discount, forward, or hazard curves (returns **`self`** for chaining). **`insert_fx(fx)`** attaches the matrix. Retrieve curves by id with **`get_discount`**, **`get_forward`**, **`get_hazard`**.

In [ ]:
from datetime import date

from finstack.core.market_data import (
    DiscountCurve,
    ForwardCurve,
    MarketContext,
)

d = DiscountCurve(
    "USD-OIS",
    date(2024, 1, 1),
    [(0.0, 1.0), (1.0, 0.95), (5.0, 0.75)],
    day_count="act_365f",
)
f = ForwardCurve(
    "USD-SOFR",
    0.25,
    [(0.0, 0.04), (1.0, 0.045)],
    base_date=date(2024, 1, 1),
    day_count="act_360",
)

mc = MarketContext()
mc.insert(d).insert(f)
print("discount id from context:", mc.get_discount("USD-OIS").id)
print("forward id from context:", mc.get_forward("USD-SOFR").id)
print("df(1) from stored curve:", mc.get_discount("USD-OIS").df(1.0))

## Mini-example: `MarketContext` for a USD corporate bond

A typical USD fixed–float or fixed coupon bond stack needs:

- **USD-OIS** (or similar) for **risk-free / valuation discounting**
- **USD-SOFR** (3M tenor) for **floating coupon projections**
- **Issuer hazard** curve when you price **credit** (skipped if unavailable)
- **FX** if you also report notionals or covenants in EUR/GBP versus USD

We build one context, attach FX, and print a short diagnostic panel.

In [ ]:
from datetime import date

from finstack.core.currency import Currency
from finstack.core.market_data import (
    DiscountCurve,
    ForwardCurve,
    FxConversionPolicy,
    FxMatrix,
    MarketContext,
)

as_of = date(2024, 1, 1)

ois = DiscountCurve(
    "USD-OIS",
    as_of,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
sofr = ForwardCurve(
    "USD-SOFR",
    0.25,
    [(0.0, 0.052), (1.0, 0.048), (3.0, 0.045), (5.0, 0.043)],
    base_date=as_of,
    day_count="act_360",
)

mc = MarketContext()
mc.insert(ois).insert(sofr)

try:
    from finstack.core.market_data import HazardCurve

    hz = HazardCurve(
        "ACME-USD-HZD",
        as_of,
        [(1.0, 0.012), (3.0, 0.018), (5.0, 0.022), (10.0, 0.028)],
        recovery_rate=0.40,
    )
    mc.insert(hz)
    hazard_line = f"survival(5y)={mc.get_hazard('ACME-USD-HZD').survival(5.0):.6f}"
except ImportError:
    hazard_line = "HazardCurve not available — credit leg omitted"

fx = FxMatrix()
fx.set_quote(Currency("EUR"), Currency("USD"), 1.10)
fx.set_quote(Currency("GBP"), Currency("USD"), 1.27)
mc.insert_fx(fx)

print("--- USD rates ---")
print("OIS df(3y):", mc.get_discount("USD-OIS").df(3.0))
print("OIS zero(3y):", mc.get_discount("USD-OIS").zero(3.0))
print("SOFR fwd(2y):", mc.get_forward("USD-SOFR").rate(2.0))
print("--- Credit ---")
print(hazard_line)
print("--- FX (vs USD) ---")
assert mc.fx is not None
print(
    "EUR/USD:",
    mc.fx.rate(
        Currency("EUR"), Currency("USD"), as_of, FxConversionPolicy.CASHFLOW_DATE
    ).rate,
)
print(
    "GBP/USD:",
    mc.fx.rate(
        Currency("GBP"), Currency("USD"), as_of, FxConversionPolicy.CASHFLOW_DATE
    ).rate,
)

## Takeaways

- **Curves** are built from **knots** in **year fractions** from a **base date**; pick **`day_count`** to match the convention of the data source.
- **`DiscountCurve.df`** and **`.zero`** are the main discounting hooks; **`ForwardCurve.rate`** gives forward rates for the curve’s fixed **reset tenor**.
- **`MarketContext`** centralizes **`insert`** for curves and **`insert_fx`** for **`FxMatrix`**; pricers retrieve by **string id** via **`get_*`**.
- **FX** requires an explicit **stored pair** (or inverse); **do not assume** automatic multi-hop triangulation unless your version documents it.
- For **corporate credit**, add a **`HazardCurve`** when the binding is available, then use **`get_hazard`** alongside OIS and SOFR for a complete USD stack.

Next in the curriculum: use this context as input to cashflow generation and instrument pricing notebooks.